# Test Semantic Search - Knowledge Base Integration

Test các components của semantic search integration:
1. MongoDB connection
2. Elasticsearch connection
3. Embedding service
4. Knowledge search service
5. LangChain tool
6. End-to-end với ChatService

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

In [ ]:
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
print("Loaded .env")

In [ ]:
import os
print("ENV VALUE:", os.getenv("KNOWLEDGE_MONGO_URI"))

In [ ]:
from app.core.config import settings

print("Config loaded:")
print(f"  MongoDB URI: {settings.KNOWLEDGE_MONGO_URI}")
print(f"  MongoDB DB: {settings.KNOWLEDGE_MONGO_DB}")
print(f"  Elasticsearch URL: {settings.KNOWLEDGE_ELASTICSEARCH_URL}")
print(f"  Elasticsearch Index: {settings.KNOWLEDGE_ELASTICSEARCH_INDEX}")
print(f"  Embedding Model: {settings.OPENAI_EMBEDDING_MODEL}")
print(f"  Top K: {settings.KNOWLEDGE_SEARCH_TOP_K}")

## 2. Test MongoDB Connection

In [ ]:
from pymongo import MongoClient

try:
    client = MongoClient(settings.KNOWLEDGE_MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    print("MongoDB: Connected successfully")
    
    db = client[settings.KNOWLEDGE_MONGO_DB]
    count = db.metadata.count_documents({})
    print(f"  Database: {settings.KNOWLEDGE_MONGO_DB}")
    print(f"  Documents in 'metadata' collection: {count}")
    
    if count > 0:
        sample = db.metadata.find_one()
        print(f"  Sample document:")
        print(f"    - doc_id: {sample.get('doc_id', 'N/A')}")
        print(f"    - title: {sample.get('title', 'N/A')}")
        print(f"    - source_type: {sample.get('source_type', 'N/A')}")
except Exception as e:
    print(f"MongoDB: Connection failed - {e}")

## 2.5. Check Elasticsearch Client Version

In [ ]:
import elasticsearch

# Get version - handle both tuple and string formats
version = elasticsearch.__version__
if isinstance(version, tuple):
    version_str = '.'.join(map(str, version))
    major_version = version[0]
else:
    version_str = str(version)
    version_parts = version_str.split('.')
    major_version = int(version_parts[0])

print(f"Elasticsearch client version: {version_str}")

# Check if version is compatible with ES 8.11.0
if major_version >= 9:
    print("\n ERROR: Elasticsearch client version 9.x is NOT compatible with ES 8.11.0")
    print("\n Please run these commands in terminal:")
    print("   pip uninstall elasticsearch -y")
    print("   pip install elasticsearch==8.11.0")
    print("\n Then RESTART KERNEL (Kernel → Restart) and rerun all cells")
    raise ImportError(
        "Elasticsearch client version mismatch. "
        "Please reinstall: pip install elasticsearch==8.11.0"
    )
else:
    print("✓ Version is compatible with ES 8.11.0")

## 3. Test Elasticsearch Connection

In [ ]:
from elasticsearch import Elasticsearch
import traceback

print("ES_URL   =", settings.KNOWLEDGE_ELASTICSEARCH_URL)
print("ES_INDEX =", settings.KNOWLEDGE_ELASTICSEARCH_INDEX)

try:
    # Không set headers, để client tự handle compatibility
    es = Elasticsearch(
        hosts=[settings.KNOWLEDGE_ELASTICSEARCH_URL],
        request_timeout=30,
        verify_certs=False,
        ssl_show_warn=False
    )

    info = es.info()
    print("Elasticsearch: Connected successfully")
    print("  Version :", info.get("version", {}).get("number"))
    print("  Cluster :", info.get("cluster_name"))
    print("  Node   :", info.get("name"))

    index_name = settings.KNOWLEDGE_ELASTICSEARCH_INDEX
    index_exists = es.indices.exists(index=index_name)
    print(f"  Index '{index_name}' exists:", index_exists)

    if index_exists:
        chunk_count = es.count(index=index_name)["count"]
        print("  Total chunks:", chunk_count)

        if chunk_count > 0:
            sample_result = es.search(
                index=index_name,
                size=1,
                query={"match_all": {}},
                _source=["doc_id", "title", "content"]
            )
            if sample_result["hits"]["hits"]:
                hit = sample_result["hits"]["hits"][0]["_source"]
                print("  Sample:")
                print("    - doc_id:", hit.get("doc_id"))
                print("    - title :", hit.get("title"))
                print("    - content length:", len(hit.get("content", "") or ""))

except Exception as e:
    print("Elasticsearch: Connection failed -", repr(e))
    traceback.print_exc()


## 4. Test Embedding Service

In [ ]:
from app.core.features.embedding import generate_embedding

test_query = "Kỹ sư phần mềm cần học những gì?"

try:
    embedding = generate_embedding(test_query)
    print(f"Embedding generated successfully")
    print(f"  Query: {test_query}")
    print(f"  Dimension: {len(embedding)}")
    print(f"  First 5 values: {embedding[:5]}")
    print(f"  Expected dimension for {settings.OPENAI_EMBEDDING_MODEL}: 1536")
    
    if len(embedding) == 1536:
        print("  Dimension matches expected value")
    else:
        print(f"  Warning: Expected 1536, got {len(embedding)}")
except Exception as e:
    print(f"Embedding generation failed: {e}")

## 5. Test Knowledge Search Service

In [ ]:
from app.core.features.knowledge_search import search_knowledge_base

test_queries = [
    "Kỹ sư phần mềm cần học những gì?",
    "Ngành AI có những chuyên ngành nào?",
    "Career path của data scientist như thế nào?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    
    try:
        results = search_knowledge_base(query, top_k=3)
        
        if results:
            print(f"Found {len(results)} results:")
            for i, result in enumerate(results, 1):
                print(f"\n[{i}] {result.get('title', 'Unknown')}")
                print(f"    Content: {result.get('content', '')[:200]}...")
                print(f"    Source: {result.get('source', 'Unknown')}")
                print(f"    Doc ID: {result.get('doc_id', 'Unknown')}")
                print(f"    Score: {result.get('score', 0):.3f}")
        else:
            print("No results found")
    except Exception as e:
        print(f"Search failed: {e}")
        import traceback
        traceback.print_exc()

## 6. Test LangChain Tool

In [ ]:
from app.core.features.knowledge_search import search_knowledge_base_tool

test_query = "Kỹ sư phần mềm cần học những gì?"

print(f"Testing tool with query: {test_query}\n")
print("-" * 60)

try:
    result = search_knowledge_base_tool.invoke({"query": test_query})
    print(result)
except Exception as e:
    print(f"Tool execution failed: {e}")
    import traceback
    traceback.print_exc()

## 7. Test với ChatService (End-to-End)

Test semantic search với ChatService - LLM sẽ tự động gọi knowledge base tool khi cần.

In [ ]:
from app.services.chat_service import get_chat_service
import uuid

chat = get_chat_service()
THREAD_ID = f"test_semantic_{uuid.uuid4().hex[:8]}"

print(f"ChatService initialized!")
print(f"Thread ID: {THREAD_ID}\n")

### Test 1: Câu hỏi về kỹ năng (sẽ dùng Knowledge Base)

In [ ]:
response1 = chat.chat(
    message="Kỹ sư phần mềm cần học những gì?",
    thread_id=THREAD_ID
)

print("=== KỸ NĂNG KỸ SƯ PHẦN MỀM ===")
print(response1)

### Test 2: Câu hỏi về chuyên ngành (sẽ dùng Knowledge Base)

In [ ]:
response2 = chat.chat(
    message="Ngành AI có những chuyên ngành nào?",
    thread_id=THREAD_ID
)

print("=== CHUYÊN NGÀNH AI ===")
print(response2)

### Test 3: Câu hỏi về career path (sẽ dùng Knowledge Base)

In [ ]:
response3 = chat.chat(
    message="Career path của data scientist như thế nào?",
    thread_id=THREAD_ID
)

print("=== CAREER PATH DATA SCIENTIST ===")
print(response3)

## 8. So sánh Knowledge Base vs Tavily Search

Test xem LLM có biết khi nào dùng knowledge base vs Google search không.

In [ ]:
from app.services.chat_service import get_chat_service
import uuid

chat_comparison = get_chat_service()
THREAD_ID_COMPARISON = f"test_comparison_{uuid.uuid4().hex[:8]}"

print(f"Thread ID: {THREAD_ID_COMPARISON}\n")

### Test 1: Query nên dùng Knowledge Base

In [ ]:
response_kb = chat_comparison.chat(
    message="Kỹ sư phần mềm cần học những kỹ năng gì?",
    thread_id=THREAD_ID_COMPARISON
)

print("=== KNOWLEDGE BASE SEARCH ===")
print(response_kb)

### Test 2: Query nên dùng Tavily Search

In [ ]:
response_google = chat_comparison.chat(
    message="Điểm chuẩn Đại học Bách Khoa năm 2024 là bao nhiêu?",
    thread_id=THREAD_ID_COMPARISON
)

print("=== GOOGLE SEARCH ===")
print(response_google)

## 9. E2E: Load profile + Holland từ file và chat tuần tự (1 session)

Giống `chatbot_demo.ipynb`: load data từ `data/`, chat nhiều turn trong cùng `THREAD_ID`, rồi in ra history.

In [ ]:
from app.services.chat_service import get_chat_service
from app.core.memory.long_term import load_profile_from_file, load_holland_from_file

# Load data
profile_data = load_profile_from_file("User_01.json")
holland_data = load_holland_from_file("Holland_test_u1.json")

if profile_data:
    print(f"Loaded profile: {profile_data.basic_info.full_name}")
else:
    raise RuntimeError("Failed to load profile_data from data/user/User_01.json")

if holland_data:
    print(f"Loaded Holland: {holland_data.top_code} (Top3: {', '.join(holland_data.top3)})")
else:
    raise RuntimeError("Failed to load holland_data from data/test/Holland_test_u1.json")

# Init ChatService and thread
chat_multi = get_chat_service()
THREAD_ID_MULTI = "semantic_search_e2e_multi"
print(f"\nChatService initialized! Thread ID: {THREAD_ID_MULTI}")

### Turn 1: Tư vấn tổng quan (dùng profile + Holland)

In [ ]:
resp_t1 = chat_multi.chat(
    message="Chào bạn, dựa vào profile và kết quả Holland của mình, bạn gợi ý giúp mình 2-3 hướng nghề phù hợp nhất và vì sao?",
    thread_id=THREAD_ID_MULTI,
    profile_data=profile_data,
    holland_data=holland_data,
)
print("=== TURN 1: TƯ VẤN TỔNG QUAN ===")
print(resp_t1)

### Turn 2: Kỹ năng cần học (3 tháng)

In [ ]:
resp_t2 = chat_multi.chat(
    message="Với hướng bạn gợi ý số 1, mình nên học những kỹ năng gì trong 3 tháng tới để tiến nhanh?",
    thread_id=THREAD_ID_MULTI,
    profile_data=profile_data,
    holland_data=holland_data,
)
print("=== TURN 2: KỸ NĂNG 3 THÁNG ===")
print(resp_t2)

### Turn 3: Lộ trình 6 tháng

In [ ]:
resp_t3 = chat_multi.chat(
    message="Bạn giúp mình lên lộ trình 6 tháng (theo tuần) gồm học gì + làm project gì + cách chứng minh năng lực?",
    thread_id=THREAD_ID_MULTI,
    profile_data=profile_data,
    holland_data=holland_data,
)
print("=== TURN 3: LỘ TRÌNH 6 THÁNG ===")
print(resp_t3)

### Turn 4: Tìm kiếm trong Knowledge Base

In [ ]:
resp_t4 = chat_multi.chat(
    message="Có tài liệu nào nói rõ về career path hoặc kỹ năng cho hướng này không? Tóm tắt giúp mình.",
    thread_id=THREAD_ID_MULTI,
    profile_data=profile_data,
    holland_data=holland_data,
)
print("=== TURN 4: KNOWLEDGE BASE SUMMARY ===")
print(resp_t4)

### History: xem toàn bộ session

In [ ]:
resp_t5 = chat_multi.chat(
    message="tỉ số trận bóng việt nam trung quốc hôm trước là bn",
    thread_id=THREAD_ID_MULTI,
    profile_data=profile_data,
    holland_data=holland_data,
)

print(resp_t5)

In [ ]:
session_data = chat_multi.get_history(THREAD_ID_MULTI)

if session_data:
    history = session_data.get("history", [])
    print(f"=== FULL CONVERSATION ({len(history)} messages) ===")
    print(f"File: data/history/{THREAD_ID_MULTI}.json")
    print()

    for i, msg in enumerate(history, 1):
        role = msg.get("role", "unknown")
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls", [])

        if role == "user":
            print(f"[{i}] USER: {content[:100]}...")
        elif role == "assistant" and tool_calls:
            tools = [tc.get("name") for tc in tool_calls]
            print(f"[{i}] ASSISTANT → TOOL CALL: {tools}")
        elif role == "tool":
            name = msg.get("name", "unknown")
            print(f"[{i}] TOOL ({name}): {len(content)} chars")
        elif role == "assistant":
            print(f"[{i}] ASSISTANT: {content[:100]}...")
else:
    print("No history found")